# 04. Hyperparameter Tuning - Grid Search

**Đồ án:** GNN Protein Function Prediction  
**Môn học:** IS353 - Mạng Xã Hội

## Mục tiêu
1. Grid Search cho hyperparameters
2. Tìm best configuration
3. Visualize hyperparameter comparison

## 1. Setup

In [ ]:
# Install dependencies
!pip install torch torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q
!pip install pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm
from itertools import product
import os
import urllib.request
import gzip
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)

## 2. Load Data (same as previous notebooks)

In [ ]:
# Load data
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/polypharmacy.csv'):
    url = 'http://snap.stanford.edu/biodata/datasets/10017/files/ChChSe-Decagon_polypharmacy.csv.gz'
    gz_path = 'data/polypharmacy.csv.gz'
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, 'rb') as f_in:
        with open('data/polypharmacy.csv', 'wb') as f_out:
            f_out.write(f_in.read())
    os.remove(gz_path)

df = pd.read_csv('data/polypharmacy.csv')
df.columns = ['Drug1', 'Drug2', 'SideEffect']

TOP_N_RELATIONS = 50
relation_counts = df['SideEffect'].value_counts()
top_relations = relation_counts.head(TOP_N_RELATIONS).index.tolist()
df_filtered = df[df['SideEffect'].isin(top_relations)].copy()

all_drugs = sorted(set(df_filtered['Drug1']) | set(df_filtered['Drug2']))
all_relations = sorted(set(df_filtered['SideEffect']))

drug_to_idx = {drug: idx for idx, drug in enumerate(all_drugs)}
relation_to_idx = {rel: idx for idx, rel in enumerate(all_relations)}

num_nodes = len(drug_to_idx)
num_relations = len(relation_to_idx)

# Build edges
edges = []
for _, row in df_filtered.iterrows():
    src = drug_to_idx[row['Drug1']]
    dst = drug_to_idx[row['Drug2']]
    rel = relation_to_idx[row['SideEffect']]
    edges.append((src, dst, rel))
    edges.append((dst, src, rel))
edges = list(set(edges))

np.random.seed(42)
np.random.shuffle(edges)
n = len(edges)
train_edges = edges[:int(0.8*n)]
val_edges = edges[int(0.8*n):int(0.9*n)]
test_edges = edges[int(0.9*n):]

def edges_to_tensors(edge_list):
    src = torch.tensor([e[0] for e in edge_list], dtype=torch.long)
    dst = torch.tensor([e[1] for e in edge_list], dtype=torch.long)
    rel = torch.tensor([e[2] for e in edge_list], dtype=torch.long)
    return torch.stack([src, dst], dim=0), rel

train_edge_index, train_edge_type = edges_to_tensors(train_edges)
val_edge_index, val_edge_type = edges_to_tensors(val_edges)
test_edge_index, test_edge_type = edges_to_tensors(test_edges)
x = torch.eye(num_nodes)

print(f"Nodes: {num_nodes}, Relations: {num_relations}")
print(f"Train: {len(train_edges)}, Val: {len(val_edges)}, Test: {len(test_edges)}")

## 3. Model Definitions

In [ ]:
class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, 
                 num_bases=None, dropout=0.0, variational=False):
        super().__init__()
        self.variational = variational
        self.dropout = dropout
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations=num_relations, num_bases=num_bases)
        self.conv2_mu = RGCNConv(hidden_channels, out_channels, num_relations=num_relations, num_bases=num_bases)
        if variational:
            self.conv2_logvar = RGCNConv(hidden_channels, out_channels, num_relations=num_relations, num_bases=num_bases)
    
    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        mu = self.conv2_mu(x, edge_index, edge_type)
        if self.variational:
            logvar = self.conv2_logvar(x, edge_index, edge_type)
            return mu, logvar
        return mu

class DistMultDecoder(nn.Module):
    def __init__(self, num_relations, embedding_dim):
        super().__init__()
        self.relation_embeddings = nn.Parameter(torch.Tensor(num_relations, embedding_dim))
        nn.init.xavier_uniform_(self.relation_embeddings)
    
    def forward(self, z, edge_index, edge_type):
        head = z[edge_index[0]]
        tail = z[edge_index[1]]
        rel = self.relation_embeddings[edge_type]
        return (head * rel * tail).sum(dim=1)

class VGAE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, num_bases=None, dropout=0.0):
        super().__init__()
        self.encoder = RGCNEncoder(in_channels, hidden_channels, out_channels, num_relations, num_bases, dropout, variational=True)
        self.decoder = DistMultDecoder(num_relations, out_channels)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def encode(self, x, edge_index, edge_type):
        mu, logvar = self.encoder(x, edge_index, edge_type)
        return self.reparameterize(mu, logvar), mu, logvar
    
    def decode(self, z, edge_index, edge_type):
        return self.decoder(z, edge_index, edge_type)
    
    def kl_loss(self, mu, logvar):
        return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))

def negative_sampling(edge_index, edge_type, num_nodes, num_neg_samples=1):
    num_edges = edge_index.size(1)
    neg_heads = edge_index[0].repeat(num_neg_samples)
    neg_types = edge_type.repeat(num_neg_samples)
    neg_tails = torch.randint(0, num_nodes, (num_edges * num_neg_samples,), device=edge_index.device)
    return torch.stack([neg_heads, neg_tails], dim=0), neg_types

## 4. Grid Search

In [ ]:
# Hyperparameter grid
param_grid = {
    'hidden_dim': [32, 64],
    'embedding_dim': [16, 32],
    'learning_rate': [0.001, 0.01],
    'dropout': [0.0, 0.3],
}

# Fixed params
NUM_BASES = 30
EPOCHS = 50  # Reduced for grid search

# Calculate total combinations
total_combinations = 1
for v in param_grid.values():
    total_combinations *= len(v)
print(f"Total hyperparameter combinations: {total_combinations}")

In [ ]:
def train_and_evaluate(hidden_dim, embedding_dim, lr, dropout, epochs=50):
    """Train VGAE with given hyperparameters and return validation AUC"""
    
    model = VGAE(
        in_channels=num_nodes,
        hidden_channels=hidden_dim,
        out_channels=embedding_dim,
        num_relations=num_relations,
        num_bases=NUM_BASES,
        dropout=dropout
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    x_dev = x.to(device)
    train_ei = train_edge_index.to(device)
    train_et = train_edge_type.to(device)
    val_ei = val_edge_index.to(device)
    val_et = val_edge_type.to(device)
    
    best_val_auc = 0
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        z, mu, logvar = model.encode(x_dev, train_ei, train_et)
        pos_scores = model.decode(z, train_ei, train_et)
        neg_ei, neg_et = negative_sampling(train_ei, train_et, num_nodes)
        neg_scores = model.decode(z, neg_ei, neg_et)
        
        pos_loss = F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores))
        neg_loss = F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores))
        kl_loss = model.kl_loss(mu, logvar)
        loss = pos_loss + neg_loss + 0.01 * kl_loss
        
        loss.backward()
        optimizer.step()
    
    # Evaluate on validation set
    model.eval()
    with torch.no_grad():
        z, _, _ = model.encode(x_dev, train_ei, train_et)
        pos_scores = model.decode(z, val_ei, val_et).sigmoid().cpu().numpy()
        neg_ei, neg_et = negative_sampling(val_ei, val_et, num_nodes)
        neg_scores = model.decode(z, neg_ei, neg_et).sigmoid().cpu().numpy()
        
        scores = np.concatenate([pos_scores, neg_scores])
        labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        val_auc = roc_auc_score(labels, scores)
    
    return val_auc, model

In [ ]:
# Run grid search
results = []
best_auc = 0
best_params = None
best_model = None

print("Running Grid Search...")
print("="*60)

for hidden, emb, lr, dropout in tqdm(list(product(
    param_grid['hidden_dim'],
    param_grid['embedding_dim'],
    param_grid['learning_rate'],
    param_grid['dropout']
))):
    torch.manual_seed(42)
    np.random.seed(42)
    
    val_auc, model = train_and_evaluate(hidden, emb, lr, dropout)
    
    results.append({
        'hidden_dim': hidden,
        'embedding_dim': emb,
        'learning_rate': lr,
        'dropout': dropout,
        'val_auc': val_auc
    })
    
    if val_auc > best_auc:
        best_auc = val_auc
        best_params = {'hidden_dim': hidden, 'embedding_dim': emb, 'lr': lr, 'dropout': dropout}
        best_model = model

# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('val_auc', ascending=False)

print("\nGrid Search Results:")
print(results_df.to_string(index=False))

In [ ]:
# Save results
results_df.to_csv('models/hyperparameter_results.csv', index=False)
print("✅ Saved: models/hyperparameter_results.csv")

# Save best model
torch.save(best_model.state_dict(), 'models/best_vgae_model.pt')
print("✅ Saved: models/best_vgae_model.pt")

print(f"\n🏆 Best Parameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")
print(f"   Val AUC: {best_auc:.4f}")

## 5. Visualization

In [ ]:
# Hyperparameter comparison bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Hidden dim
ax = axes[0, 0]
hidden_means = results_df.groupby('hidden_dim')['val_auc'].mean()
ax.bar(hidden_means.index.astype(str), hidden_means.values, color='steelblue')
ax.set_xlabel('Hidden Dimension')
ax.set_ylabel('Mean Val AUC')
ax.set_title('Effect of Hidden Dimension')

# Embedding dim
ax = axes[0, 1]
emb_means = results_df.groupby('embedding_dim')['val_auc'].mean()
ax.bar(emb_means.index.astype(str), emb_means.values, color='coral')
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Mean Val AUC')
ax.set_title('Effect of Embedding Dimension')

# Learning rate
ax = axes[1, 0]
lr_means = results_df.groupby('learning_rate')['val_auc'].mean()
ax.bar([str(x) for x in lr_means.index], lr_means.values, color='green')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Mean Val AUC')
ax.set_title('Effect of Learning Rate')

# Dropout
ax = axes[1, 1]
drop_means = results_df.groupby('dropout')['val_auc'].mean()
ax.bar([str(x) for x in drop_means.index], drop_means.values, color='purple')
ax.set_xlabel('Dropout')
ax.set_ylabel('Mean Val AUC')
ax.set_title('Effect of Dropout')

plt.tight_layout()
plt.savefig('figures/hyperparameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/hyperparameter_comparison.png")

In [ ]:
# Heatmap: hidden_dim vs embedding_dim
fig, ax = plt.subplots(figsize=(10, 6))

pivot = results_df.pivot_table(values='val_auc', index='hidden_dim', columns='embedding_dim', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax)
ax.set_title('Val AUC: Hidden Dim vs Embedding Dim', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/hyperparameter_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/hyperparameter_heatmap.png")

## 6. Final Test with Best Model

In [ ]:
# Evaluate best model on test set
best_model.eval()
x_dev = x.to(device)
test_ei = test_edge_index.to(device)
test_et = test_edge_type.to(device)
train_ei = train_edge_index.to(device)
train_et = train_edge_type.to(device)

with torch.no_grad():
    z, _, _ = best_model.encode(x_dev, train_ei, train_et)
    pos_scores = best_model.decode(z, test_ei, test_et).sigmoid().cpu().numpy()
    neg_ei, neg_et = negative_sampling(test_ei, test_et, num_nodes)
    neg_scores = best_model.decode(z, neg_ei, neg_et).sigmoid().cpu().numpy()
    
    scores = np.concatenate([pos_scores, neg_scores])
    labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    
    test_auc = roc_auc_score(labels, scores)
    test_ap = average_precision_score(labels, scores)

print("="*60)
print("BEST MODEL - TEST RESULTS")
print("="*60)
print(f"Best Hyperparameters: {best_params}")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"Test AP: {test_ap:.4f}")

---

## ✅ Checklist Phase 4 (Grid Search)

- [x] Define hyperparameter grid
- [x] Run grid search
- [x] Find best hyperparameters
- [x] Visualize hyperparameter effects
- [x] Save best model
- [x] Test with best model

**Next:** Phase 5 - Evaluation (ROC for top 5 relations, t-SNE, Interpretation)